# Auto-Labeling for Trace Triage

This notebook labels failed traces with the Trace Triage recovery taxonomy.

It reads:

```text
data/labeling_exports/failed_traces_non_ablation.jsonl
```

and writes resumable labels to:

```text
data/labeling_exports/[model]_auto_labels.jsonl
```

By default, it labels only rows where `needs_labeling = 1`, meaning CausalFlow did **not** auto-label the trace as `LOCAL_REPAIR`.


## Setup

This notebook is configured for **OpenRouter** using the OpenAI-compatible Python SDK.

If `openai` is not installed in your notebook kernel, run the install cell. Then set your OpenRouter API key:

```python
import os
os.environ["OPENROUTER_API_KEY"] = "..."
```

Do not save real API keys directly in the notebook.


In [88]:
# Run this only if your notebook kernel does not already have these packages.
# %pip install -U openai pydantic tqdm

In [89]:
from pathlib import Path
from typing import Literal
import json
import os
import sqlite3
import time

from pydantic import BaseModel, Field
from tqdm.auto import tqdm

try:
    from openai import OpenAI
except ImportError as exc:
    raise ImportError("Install the OpenAI SDK first: %pip install -U openai") from exc


In [90]:
models = {'llama': 'meta-llama/llama-3.3-70b-instruct', 'gpt': 'openai/gpt-oss-120b'}
model = 'gpt'

In [91]:
# Configuration
INPUT_JSONL = Path("data/labeling_exports/failed_traces.jsonl")
OUTPUT_JSONL = Path(f"data/labeling_exports/{model}_auto_labels.jsonl")
ERROR_JSONL = Path(f"data/labeling_exports/{model}_auto_label_errors.jsonl")
DB_PATH = Path("data/causal_runs.sqlite")

# OpenRouter settings. Pick any OpenRouter model you have access to.
# Examples may include models available through OpenRouter, e.g. "openai/gpt-5.5".
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL = os.getenv("OPENROUTER_MODEL", models[model])

# Optional OpenRouter metadata headers.
OPENROUTER_SITE_URL = os.getenv("OPENROUTER_SITE_URL", "http://localhost")
OPENROUTER_APP_NAME = os.getenv("OPENROUTER_APP_NAME", "TraceTriage Auto Labeler")

# Auto-labeled LOCAL_REPAIR rows are written to OUTPUT_JSONL without calling the model.
# Only rows with needs_labeling = 1 are sent to the model.
INCLUDE_AUTO_LOCAL_REPAIR_IN_OUTPUT = True

# Use a small number like 5 for testing model calls, then set to None for the full run.
# This limit only applies to model-labeled rows, not auto LOCAL_REPAIR rows.
MAX_RECORDS = None

# How many trace steps to include per prompt. None means all steps, but that may cost more.
MAX_STEPS = 30
MAX_STEP_CHARS = 900
MAX_PROBLEM_CHARS = 1800
MAX_ANSWER_CHARS = 600

In [92]:
class TriagePrediction(BaseModel):
    action: Literal["RETRY", "REPLAN", "RETRIEVE_MORE", "TOOL_FIX", "ESCALATE"]
    rationale: str = Field(description="One concise sentence explaining why this recovery action is most appropriate.")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence from 0 to 1.")


def read_jsonl(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing input file: {path}. Run the export cell in analysis.ipynb first.")
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def append_jsonl(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_existing_labels(path):
    if not path.exists():
        return {}
    labels = {}
    for row in read_jsonl(path):
        labels[row["trace_id"]] = row
    return labels


def truncate(value, max_chars):
    if value is None:
        return ""
    text = str(value).strip()
    if len(text) > max_chars:
        return text[:max_chars] + "\n...[truncated]"
    return text


def allowed_actions(record):
    actions = json.loads(record.get("applicable_actions_json") or "[]")
    return [a for a in actions if a != "LOCAL_REPAIR"]


def is_auto_local_repair(record):
    return record.get("action_label") == "LOCAL_REPAIR" or record.get("is_local_repairable") == 1


def make_local_repair_label(record):
    return {
        "trace_id": record["trace_id"],
        "problem_id": record.get("problem_id"),
        "domain": record.get("domain"),
        "provider": "causalflow",
        "model": None,
        "action": "LOCAL_REPAIR",
        "rationale": "CausalFlow found at least one successful local counterfactual repair for this failed trace.",
        "confidence": 1.0,
        "allowed_actions": json.loads(record.get("applicable_actions_json") or "[]"),
        "label_source": "auto_causalflow",
    }


In [93]:
def compact_trace_text(record):
    lines = []
    steps = record.get("steps", [])
    if MAX_STEPS is not None:
        steps = steps[:MAX_STEPS]

    for step in steps:
        header = f"Step {step.get('step_index')} | id={step.get('step_id')} | type={step.get('step_type')}"
        if step.get("tool_name"):
            header += f" | tool={step.get('tool_name')}"
        lines.append(header)

        if step.get("tool_args_json"):
            lines.append("Tool args: " + truncate(step.get("tool_args_json"), 400))

        if step.get("text"):
            lines.append("Text: " + truncate(step.get("text"), MAX_STEP_CHARS))

        if step.get("tool_output_json"):
            lines.append("Tool output: " + truncate(step.get("tool_output_json"), MAX_STEP_CHARS))

        lines.append("")

    if MAX_STEPS is not None and len(record.get("steps", [])) > MAX_STEPS:
        lines.append(f"...[{len(record['steps']) - MAX_STEPS} additional steps omitted]")

    return "\n".join(lines)


def build_prompt(record):
    actions = allowed_actions(record)
    return f"""
You are labeling a failed AI agent trace for the Trace Triage recovery-action task.

Choose the recovery action most likely to fix the failed trace.

Task domain: {record.get('domain')}
Problem ID: {record.get('problem_id')}
Applicable actions for this domain: {actions}

Action definitions:
- RETRY: The failure looks like a stochastic mistake, formatting issue, or minor instability. Resampling the same strategy may fix it.
- REPLAN: The agent's strategy or plan was wrong from the start. It needs a different approach, not a local tweak.
- RETRIEVE_MORE: The failure is due to missing, weak, or insufficient external evidence. More/better retrieval is likely needed.
- TOOL_FIX: The failure is due to wrong tool choice, bad tool arguments, tool execution errors, or mishandled tool use.
- ESCALATE: The failure is ambiguous, unsafe, unverifiable, or unlikely to be fixed automatically.

Important:
- Do not choose LOCAL_REPAIR here; CausalFlow local repairs were already auto-labeled separately.
- Choose only from the applicable actions listed above.
- Focus on what recovery action should be tried next, not merely which step was wrong.

Problem:
{truncate(record.get('problem_statement'), MAX_PROBLEM_CHARS)}

Correct answer:
{truncate(record.get('gold_answer'), MAX_ANSWER_CHARS)}

Agent's wrong final answer:
{truncate(record.get('final_answer'), MAX_ANSWER_CHARS)}

Agent execution trace:
{compact_trace_text(record)}
""".strip()


In [94]:
# Set your OpenRouter key before running the labeling cells.
# Prefer setting it outside the notebook, but this is okay temporarily during local testing:
# os.environ["OPENROUTER_API_KEY"] = "sk-or-v1-..."

import os
os.environ["OPENROUTER_API_KEY"] = "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"
os.environ["OPENROUTER_MODEL"] = models[model]

if not os.getenv("OPENROUTER_API_KEY"):
    raise RuntimeError("Set OPENROUTER_API_KEY before calling OpenRouter.")


In [95]:
client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url=OPENROUTER_BASE_URL,
    default_headers={
        "HTTP-Referer": OPENROUTER_SITE_URL,
        "X-Title": OPENROUTER_APP_NAME,
    },
)

SYSTEM_INSTRUCTIONS = """You are an expert annotator for failed LLM-agent traces. Return only valid JSON matching the requested schema. Be conservative and use ESCALATE when the trace is too ambiguous to diagnose."""


def parse_label_content(content):
    """Parse and validate a JSON label returned by an OpenRouter chat completion."""
    if content is None:
        raise ValueError("Model returned empty content")

    text = content.strip()
    if text.startswith("```json"):
        text = text.removeprefix("```json").strip()
    if text.startswith("```"):
        text = text.removeprefix("```").strip()
    if text.endswith("```"):
        text = text[:-3].strip()

    data = json.loads(text)
    return TriagePrediction(**data)


def call_model_label(record, max_retries=3):
    actions = allowed_actions(record)
    prompt = build_prompt(record)

    user_message = prompt + """

Return strictly valid JSON with this shape:
{
  "action": "RETRY | REPLAN | RETRIEVE_MORE | TOOL_FIX | ESCALATE",
  "rationale": "one concise sentence",
  "confidence": 0.0
}
"""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTIONS},
                    {"role": "user", "content": user_message},
                ],
                temperature=0,
                response_format={"type": "json_object"},
            )
            pred = parse_label_content(response.choices[0].message.content)

            if pred.action not in actions:
                raise ValueError(f"Model returned action {pred.action}, not in allowed actions {actions}")

            return {
                "trace_id": record["trace_id"],
                "problem_id": record.get("problem_id"),
                "domain": record.get("domain"),
                "provider": "openrouter",
                "model": MODEL,
                "action": pred.action,
                "rationale": pred.rationale,
                "confidence": pred.confidence,
                "allowed_actions": actions,
            }
        except Exception as exc:
            if attempt == max_retries - 1:
                raise
            time.sleep(10 * (2 ** attempt))


In [96]:
records = list(read_jsonl(INPUT_JSONL))
print(f"Loaded {len(records)} records from {INPUT_JSONL}")

existing = load_existing_labels(OUTPUT_JSONL)

if INCLUDE_AUTO_LOCAL_REPAIR_IN_OUTPUT:
    auto_records = [r for r in records if is_auto_local_repair(r)]
    auto_to_write = [r for r in auto_records if r["trace_id"] not in existing]
    for record in auto_to_write:
        append_jsonl(OUTPUT_JSONL, make_local_repair_label(record))
    if auto_to_write:
        existing = load_existing_labels(OUTPUT_JSONL)
else:
    auto_records = []
    auto_to_write = []

# Only unresolved traces are sent to the model.
records_to_label = [
    r for r in records
    if r.get("needs_labeling") == 1
    and not is_auto_local_repair(r)
    and r["trace_id"] not in existing
]

if MAX_RECORDS is not None:
    records_to_label = records_to_label[:MAX_RECORDS]

print(f"Auto LOCAL_REPAIR records in input: {len(auto_records)}")
print(f"Auto LOCAL_REPAIR records newly written: {len(auto_to_write)}")
print(f"Already present in output: {len(existing)}")
print(f"Will send to model now: {len(records_to_label)}")


Loaded 638 records from data/labeling_exports/failed_traces.jsonl
Auto LOCAL_REPAIR records in input: 0
Auto LOCAL_REPAIR records newly written: 0
Already present in output: 0
Will send to model now: 638


In [97]:
# Run model labeling only for unresolved traces. Auto LOCAL_REPAIR rows were already written above.
for record in tqdm(records_to_label):
    try:
        label = call_model_label(record)
        append_jsonl(OUTPUT_JSONL, label)
        time.sleep(5)
    except Exception as exc:
        error = {
            "trace_id": record.get("trace_id"),
            "problem_id": record.get("problem_id"),
            "domain": record.get("domain"),
            "error": repr(exc),
        }
        append_jsonl(ERROR_JSONL, error)
        print("Error:", error)

print(f"Complete labels written to {OUTPUT_JSONL}")
print(f"Errors written to {ERROR_JSONL}")

  0%|          | 0/638 [00:00<?, ?it/s]

Complete labels written to data/labeling_exports/gpt_auto_labels.jsonl
Errors written to data/labeling_exports/gpt_auto_label_errors.jsonl


In [98]:
# Preview label distribution so far.
labels = list(read_jsonl(OUTPUT_JSONL)) if OUTPUT_JSONL.exists() else []
from collections import Counter

print(f"Total {model} labels:", len(labels))
print("By action:", Counter(row["action"] for row in labels))
print("By domain:", Counter(row["domain"] for row in labels))
labels[:3]

Total gpt labels: 638
By action: Counter({'RETRIEVE_MORE': 275, 'REPLAN': 174, 'TOOL_FIX': 111, 'RETRY': 75, 'ESCALATE': 3})
By domain: Counter({'MBPP': 259, 'MedBrowseComp': 235, 'SealQA': 120, 'GSM8K': 21, 'BrowseComp': 3})


[{'trace_id': 'run_BrowseComp_2025-12-15T20:38:47.779542::browsecomp_3',
  'problem_id': 'browsecomp_3',
  'domain': 'BrowseComp',
  'provider': 'openrouter',
  'model': 'openai/gpt-oss-120b',
  'action': 'RETRIEVE_MORE',
  'rationale': 'The agent failed to find the specific blog post containing the track list, so additional or refined retrieval is needed.',
  'confidence': 0.86,
  'allowed_actions': ['RETRY',
   'REPLAN',
   'RETRIEVE_MORE',
   'TOOL_FIX',
   'ESCALATE']},
 {'trace_id': 'run_BrowseComp_2025-12-15T20:38:47.779542::browsecomp_6',
  'problem_id': 'browsecomp_6',
  'domain': 'BrowseComp',
  'provider': 'openrouter',
  'model': 'openai/gpt-oss-120b',
  'action': 'RETRIEVE_MORE',
  'rationale': 'The agent failed to locate the correct article, indicating insufficient or missing external evidence.',
  'confidence': 0.9,
  'allowed_actions': ['RETRY',
   'REPLAN',
   'RETRIEVE_MORE',
   'TOOL_FIX',
   'ESCALATE']},
 {'trace_id': 'run_BrowseComp_2025-12-15T20:38:47.779542::brow